# osu-ai-dev: обучение BC на бесплатном GPU Colab

Ноутбук гоняет `bc_train.py` (сбор в симе + BC + прогрев критика + перенос в PPO)
на GPU Colab (T4), а домой ты забираешь только **два файла весов**. Дальше всё
как обычно на твоём компе: `train_stage --resume`, play, dagger — веса ничем
не отличаются от обученных локально (это просто тензоры; стек читается из самих весов).

## Что приготовить заранее (2 файла)

1. **Свежий `osu-ai-dev-stage3.zip`** — с фиксом 3.2.2 (ноутбук это проверит:
   в `bc_train.py` должен быть `including_emulation`).
2. **`sim_maps.zip`** — твои карты. В PowerShell из корня osu-ai-dev:

   ```
   Compress-Archive -Path sim_maps, sim_maps_t1 -DestinationPath sim_maps.zip
   ```

   (внутри архива должны быть папки `sim_maps/` и `sim_maps_t1/` с .osu-файлами)

## Ориентиры времени на T4

- короткий тест (ячейка 4): ~2–3 мин;
- основной прогон 250k шагов: **~60–90 мин** в fp32 на T4 (кривая обучения
  выходит на плато после ~150k — 250k достаточно; полный 500k ≈ вдвое
  дольше, если захочешь).

## Жёлтые предупреждения в начале — это НЕ ошибки

- «Gym has been unmaintained since 2022…» — прощальное письмо старого пакета
  `gym`, который лежит в образе Colab с давних времён и в нашем конвейере не
  используется: код проекта и sb3 работают только с `gymnasium` (он как раз
  установлен). Игнорируй.
- «absl … cudart_stub.cc … GPU will not be used» — TensorFlow из образа Colab
  сообщает, что не нашёл драйверов ДЛЯ СЕБЯ. Наш процесс GPU прекрасно видит
  (`device: cuda` + буфер кадров в VRAM) — на прогон не влияет.
- «jupyter_client … datetime.utcnow() … deprecated» — внутренности самих
  ноутбуков Colab, к обучению отношения не имеет.

Ошибка — это красный блок с Traceback. Всё жёлтое — шум.

Не закрывай вкладку во время основного прогона. Если Colab отключится —
перезапусти с ячейки 1 (данные теряются, веса пишутся только в конце).


In [ ]:
import torch

assert torch.cuda.is_available(), (
    "Нет GPU: меню 'Среда выполнения' -> 'Сменить среду выполнения' -> T4 GPU, затем снова")
p = torch.cuda.get_device_properties(0)
free_b, total_b = torch.cuda.mem_get_info()
cap = torch.cuda.get_device_capability(0)
print(f"GPU: {p.name} | VRAM {total_b / 2**30:.1f} ГБ, свободно {free_b / 2**30:.1f} ГБ")
print(f"compute capability sm_{cap[0]}{cap[1]} → железный bf16: {cap[0] >= 8}")
print("(T4 = sm_75: bf16 там только эмуляция без ускорения — код 3.2.2 сам")
print(" оставит fp32 + буфер в VRAM, это норма)")

In [ ]:
import glob
import os
import shutil
import zipfile

from google.colab import files

print("Выбери ДВА архива: osu-ai-dev-stage3.zip и sim_maps.zip")
up = files.upload()
os.makedirs("/content/up", exist_ok=True)
for name, blob in up.items():
    open(f"/content/up/{name}", "wb").write(blob)

os.makedirs("/content/osu-ai-dev", exist_ok=True)
for name in up:
    tmp = f"/content/up/x_{name}"
    os.makedirs(tmp, exist_ok=True)
    with zipfile.ZipFile(f"/content/up/{name}") as z:
        z.extractall(tmp)
    # это архив проекта? (внутри есть bc_train.py)
    hits = glob.glob(f"{tmp}/**/bc_train.py", recursive=True)
    if hits:
        src = os.path.dirname(hits[0])
        for f in os.listdir(src):
            dst = os.path.join("/content/osu-ai-dev", f)
            if os.path.exists(dst) and os.path.isdir(dst):
                shutil.rmtree(dst)
            shutil.move(os.path.join(src, f), dst)
        print(f"[ok] {name}: код проекта распакован")
        continue
    # иначе ищем папки карт
    found = False
    for d in ("sim_maps", "sim_maps_t1"):
        for h in [p for p in glob.glob(f"{tmp}/**/{d}", recursive=True) if os.path.isdir(p)]:
            dst = f"/content/osu-ai-dev/{d}"
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.move(h, dst)
            print(f"[ok] {name}: {d} — {len(glob.glob(dst + '/*.osu'))} карт")
            found = True
    if not found:
        # россыпь .osu прямо в корне архива без папок
        loose = glob.glob(f"{tmp}/*.osu") + glob.glob(f"{tmp}/**/*.osu", recursive=True)
        if loose and not glob.glob("/content/osu-ai-dev/sim_maps/*.osu"):
            os.makedirs("/content/osu-ai-dev/sim_maps", exist_ok=True)
            for f in loose:
                shutil.move(f, "/content/osu-ai-dev/sim_maps/")
            print(f"[ok] {name}: {len(loose)} .osu положены в sim_maps/")

# проверки
assert os.path.exists("/content/osu-ai-dev/bc_train.py"), (
    "проект не найден в загрузках — нужен osu-ai-dev-stage3.zip")
src = open("/content/osu-ai-dev/bc_train.py").read()
assert "including_emulation" in src, (
    "СТАРЫЙ архив (без фикса 3.2.2 — на T4 включалась bf16-эмуляция и резала скорость x2) —"
    " перекачай osu-ai-dev-stage3.zip и начни с этой ячейки заново")
n_maps = (len(glob.glob("/content/osu-ai-dev/sim_maps/*.osu"))
          + len(glob.glob("/content/osu-ai-dev/sim_maps_t1/*.osu")))
print(f"\nкарт всего: {n_maps}")
assert n_maps > 0, "нет .osu карт — проверь sim_maps.zip (папки sim_maps/ и sim_maps_t1/)"
print("готово. дальше — зависимости")

In [ ]:
# ВАЖНО: НЕ ставь requirements.txt целиком в Colab (torch трогать нельзя —
# сломается GPU-сборка Colab). Ставим только то, чего не хватает.
!pip install -q "stable-baselines3==2.9.2"

import cv2
import gymnasium
import numpy
import stable_baselines3

print("sb3", stable_baselines3.__version__, "| cv2", cv2.__version__,
      "| gymnasium", gymnasium.__version__, "| numpy", numpy.__version__)

## Ячейка 4 — короткий тест конвейера (~2–3 мин)

Проверяет весь путь (сбор -> BC -> прогрев -> перенос -> самопроверка) на
масштабе 5000 шагов. В конце должно быть «перенос чист», а строки «тайминг
мс/шаг» покажут скорость T4. Если тут всё чисто — гоняем основной прогон.

In [ ]:
%cd /content/osu-ai-dev
!python bc_train.py --steps 5000 --warmup 2000 --out bc_colab_test --timing

## Ячейка 5 — основной прогон

250k шагов + прогрев 75k (плато кривой ~150k). Полный вариант — без
`--steps/--warmup` (500k/150k, вдвое дольше). Ждать «прогрев завершён ->
сохранено -> перенос чист». На T4 в логе НЕ должно быть строки `[amp]` —
код 3.2.2 сам держит fp32 (появилась — архив старый, скорость будет x2 хуже).

In [ ]:
%cd /content/osu-ai-dev
!python bc_train.py --steps 250000 --warmup 75000 --out bc_policy_fresh --timing

In [ ]:
%cd /content/osu-ai-dev
# проверке GPU не нужен; прячем CUDA — работает с любой версией скрипта
!CUDA_VISIBLE_DEVICES="" python verify_transfer.py bc_policy_fresh

In [ ]:
from google.colab import files

%cd /content/osu-ai-dev
files.download("bc_policy_fresh.zip")
files.download("bc_policy_fresh_bcnet.pt")

## Дальше на своём компе — можно? ДА

Веса (`bc_policy_fresh.zip` + `bc_policy_fresh_bcnet.pt`) — это просто файлы
тензоров, они не привязаны ни к GPU, ни к ОС, ни к Colab:

1. положи оба файла в корень `osu-ai-dev` на своём компе;
2. дальше всё как с локально обученными: `train_stage --resume bc_policy_fresh.zip`,
   play, dagger — стек кадров читается из самих весов, устройство подставится твоё;
3. твоя GTX 1060 для инференса и PPO-фазы хватает (PPO уже шёл на ней).

Нюансы:

- **Версии**: Colab живет на своем torch (его не трогаем), sb3 ставим 2.9.2 —
  как в тестах. Если дома `PPO.load` вдруг ругнётся на файл — пришли текст
  ошибки, лечится `pip install stable-baselines3==2.9.2`.
- **verify_transfer дома**: свежий скрипт из архива работает сам; если
  запускаешь старую копию и падает на «FloatTensor» — PowerShell:
  `$env:CUDA_VISIBLE_DEVICES=''; python verify_transfer.py bc_policy_fresh`.
- **Если Colab отключился** во время прогона — ничего не сохранилось (веса
  пишутся в самом конце): перезапусти ноутбук с ячейки 1.
- **Кривые** сравнивай с локальными по move acc на одинаковых шагах —
  датасет тот же (тот же seed сброса, те же карты), расхождение только в
  порядке батчей — это норма.
